# 🚀 Warmup Cache — Pre-generate Analysis Embeddings

Notebook này chạy **local** để pre-cache tất cả analysis embeddings trước khi training.

**Flow:**
1. Đọc config → xác định stocks & window
2. Chạy `src.fundamental.pipeline` tuần tự cho tất cả windows → cache vào `src/fundamental/st_data/`
3. Training sẽ hit cache, không cần gọi LLM API nữa

**Có thể interrupt bất cứ lúc nào** — đã cache sẽ không gọi lại.

> Trước khi chạy, đảm bảo đã cấu hình đúng: `python -m src.fundamental.check_config`

In [1]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your dataset files
upload_folder(
    folder_path="artifacts",
    path_in_repo="",
    repo_id="bnquys/stock-prediction-hyprid", 
    repo_type="dataset"
)

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/bnquys/stock-prediction-hyprid/commit/f0dbc454ff803e31c1a4f9fe243ef79921755d40', commit_message='Upload folder using huggingface_hub', commit_description='', oid='f0dbc454ff803e31c1a4f9fe243ef79921755d40', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bnquys/stock-prediction-hyprid', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bnquys/stock-prediction-hyprid'), pr_revision=None, pr_num=None)

## 1. Load Config & Kiểm tra Data

In [4]:
import logging
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from src.config import Config
from src.technical.preprocessor import load_csv
from src.fundamental.pipeline import pipeline

logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%H:%M:%S',
)

# Load config (analysis.model tự merge từ src/fundamental/config.yaml)
cfg = Config.load('configs/')

WINDOW = cfg.env['window']  # 20
MODEL = cfg.analysis['model']
PATHS = cfg.data['paths']

print(f'Window: {WINDOW}')
print(f'Model:  {MODEL}')
print(f'Stocks: {[Path(p).stem for p in PATHS]}')

# Kiểm tra data files
for p in PATHS:
    if Path(p).exists():
        df = load_csv(p)
        print(f"  ✓ {Path(p).stem}: {len(df)} rows | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()} | Windows: {len(df)-WINDOW}")
    else:
        print(f"  ✗ {p} NOT FOUND!")

Window: 20
Model:  unsloth/Qwen2.5-14B-Instruct-bnb-4bit
Stocks: ['VNM', 'FPT', 'VIC', 'HPG']
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
  ✓ VNM: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
  ✓ FPT: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
  ✓ VIC: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
  ✓ HPG: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266


## 2. Chạy Cache (Tuần tự)

Mỗi window gọi pipeline 1 lần. Pipeline tự cache theo hash nội dung report:
- Nếu report đã tồn tại → skip tạo report
- Nếu LLM response đã tồn tại → skip gọi API
- Nếu embedding đã tồn tại → skip tính embedding

→ Chạy lại an toàn, chỉ tốn thời gian cho windows chưa cache.

In [5]:
def warmup_stock(
    stock_id: str,
    csv_path: str,
    window: int = 20,
    skip_every: int = 1,
):
    """
    Cache tất cả windows cho 1 stock, xử lý tuần tự từng window một.

    Args:
        skip_every: Cache mỗi N windows (1 = tất cả, 5 = mỗi 5 ngày).
    """
    df = load_csv(csv_path)

    success_count = 0
    errors = 0
    error_msgs = []

    steps = list(range(window, len(df), skip_every))
    pbar = tqdm(steps, desc=stock_id, unit="win")

    for step in pbar:
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()

        try:
            pipeline(
                model=MODEL,
                stock_id=stock_id,
                date_start=date_start,
                date_end=date_end,
            )
            success_count += 1
        except Exception as e:
            logging.error(e)
            errors += 1
            if len(error_msgs) < 5:
                error_msgs.append(f"date={date_end.date()}: {e}")

        pbar.set_postfix(ok=success_count, err=errors)

    pbar.close()

    # Tóm tắt
    print(f"  ✓ {stock_id}: {success_count} ok, {errors} errors (total: {len(steps)})")
    if error_msgs:
        print(f"  Errors (showing first {len(error_msgs)}):")
        for msg in error_msgs:
            print(f"    • {msg}")

    return success_count, errors

## 3. Chạy từng stock (có thể interrupt an toàn)

In [ ]:
warmup_stock("VNM", p, WINDOW, skip_every=1)

[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65


VNM:   0%|          | 0/1266 [00:00<?, ?win/s]

11:01:27 WARNING [CKey] Attempt 1/3 failed: 404 Client Error: Not Found for url: https://ckey.vn/v1/chat/completions
11:01:32 WARNING [CKey] Attempt 2/3 failed: 404 Client Error: Not Found for url: https://ckey.vn/v1/chat/completions
